Carga de importaciones y modelo beto.

In [ ]:
import torch
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification)
from torch.utils.data import DataLoader

RUTA_MODELO = r"Z:\00 COMUN PRESTACIONES Y PROVEEDORES\NPS_POWER_BI\Copias\00_NPS\Pruebas\beto-sentiment"


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(
    RUTA_MODELO,
    local_files_only=True
)

model = AutoModelForSequenceClassification.from_pretrained(
    RUTA_MODELO,
    local_files_only=True
).to(device)

model.eval()

KeyboardInterrupt: 

Carga excel

In [ ]:
df = pd.read_excel(r"Z:\00 COMUN PRESTACIONES Y PROVEEDORES\NPS_POWER_BI\Copias\00_NPS\Pruebas\Hogar\analisis_hogar.xlsx")
df.to_csv("Datos/hogar_completo.csv", index=False, sep=",", encoding="utf-8-sig")

Limpieza y tokenización

In [ ]:
import re

df = pd.read_csv("Datos/hogar_completo.csv")

df = df.dropna(subset=["ES - Corredores - Verbatim Comment including Corredores"])
df["texto"] = df["ES - Corredores - Verbatim Comment including Corredores"].astype(str)


def normalizar_texto(texto):
    texto = texto.strip()
    texto = re.sub(r"\s+", " ", texto)   
    texto = texto.lower()                
    return texto


df["texto_norm"] = df["texto"].apply(normalizar_texto)

def tokenize_batch(textos):
    return tokenizer(
        textos,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )
batch_size = 32

loader = DataLoader(
    df["texto_norm"].tolist(),
    batch_size=batch_size
)

sentimientos = []
confianzas = []

with torch.no_grad():
    for batch in loader:
        inputs = tokenize_batch(batch).to(device)

        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)

        pred = torch.argmax(probs, dim=1)
        conf = probs.max(dim=1).values

        sentimientos.extend(pred.tolist())
        confianzas.extend(conf.tolist())

mapa = {
    0: "Negativo",
    1: "Positivo",
}

df["sentimiento"] = sentimientos
df["sentimiento_label"] = df["sentimiento"].map(mapa)
df["confianza"] = confianzas

C:\Users\GGNADI2\AppData\Local\Temp\ipykernel_24724\1912780969.py:3: DtypeWarning: Columns (0: ES - Phone number, 1: Export AT Description, 2: ES - Unit ID Text, 3: ES - Unit ID, 4: ES - Contact Intrest including Corredores, 5: ES - Home - Risk Address Town, 6: ES - Home - Risk Address Province, 7: ¿Has conseguido contactar con el cliente?, 8: ¿El cliente queda conforme?, 9: Observaciones, 10: Indique el motivo de insatisfacción detectado en el asegurado:, 11: Número Expediente Operacional, 12: Código Profesional, 13: Nombre Profesional, 14: N. Expediente, 15: CÛd. profesional asignado, 16: Nombre del proveedor asignado, 17: COD_ZONA, 18: NIVEL_2, 19: CTT, 20: ES - General segment, 21: EXPORT PRESTACIONES_DGT, 22: Export DT Description, 23: ES - General Claim - Type of claim Code, 24: ES - Home - Detailed Reason Claim Communication including Corred, 25: ES - Home - Detailed Reason Claim Evolution including Corredores, 26: ES - Detailed Reason Claim Resolution including Corr, 27: ES - D

Análisis de sentimientos

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

RUTA_EMBEDDINGS = r"C:\Users\GGNADI2\OneDrive - MAPFRE\Escritorio\Análisis_sentimiento\beto-sentiment"

tokenizer_emb = AutoTokenizer.from_pretrained(
    RUTA_EMBEDDINGS,
    local_files_only=True
)

model_emb = AutoModel.from_pretrained(
    RUTA_EMBEDDINGS,
    local_files_only=True
)

model_emb.eval()

def obtener_embedding(texto):
    texto = normalizar_texto(texto)

    inputs = tokenizer_emb(
        texto,
        truncation=True,
        padding=True,
        max_length=256,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model_emb(**inputs)

    embedding = outputs.last_hidden_state[:, 0, :].numpy()
    return embedding


def normalizar(v):
    return v / np.linalg.norm(v)

proto_positivo = [
    "la atención fue adecuada",
    "el servicio ha sido satisfactorio",
    "todo se resolvió correctamente",
    "buena atención recibida",
    "gestión correcta del caso"
]

proto_neutro = [
    "se realizó la gestión",
    "información facilitada",
    "consulta registrada",
    "sin incidencias destacables",
    "proceso estándar"
]

proto_negativo = [
    "problemas con la atención",
    "no se resolvió el problema",
    "incidencias en el servicio",
    "experiencia insatisfactoria",
    "mala gestión del caso"
]

def embedding_medio(frases):
    embs = []
    for f in frases:
        emb = obtener_embedding(f)[0]
        embs.append(normalizar(emb))
    return np.mean(embs, axis=0)

emb_pos = embedding_medio(proto_positivo)
emb_neu = embedding_medio(proto_neutro)
emb_neg = embedding_medio(proto_negativo)

df_neutros = df[df["sentimiento"] == 2].copy()

def reclasificar_neutro(texto, delta=0.08):
    emb_texto = obtener_embedding(texto)[0]
    emb_texto = normalizar(emb_texto)

    sim_pos = cosine_similarity([emb_texto], [emb_pos])[0][0]
    sim_neu = cosine_similarity([emb_texto], [emb_neu])[0][0]
    sim_neg = cosine_similarity([emb_texto], [emb_neg])[0][0]

    sims = {
        "Positivo_leve": sim_pos,
        "Neutro_real": sim_neu,
        "Negativo_leve": sim_neg
    }

    etiqueta_max = max(sims, key=sims.get)

    
    sims_sorted = sorted(sims.values(), reverse=True)
    if sims_sorted[0] - sims_sorted[1] < delta:
        return "Neutro_real", sims_sorted[0]

    return etiqueta_max, sims[etiqueta_max]

df_neutros = df[df["sentimiento"] == 2].copy()

df_neutros["sub_sentimiento"], df_neutros["sub_confianza"] = zip(
    *df_neutros["texto"].apply(reclasificar_neutro)
)

df.loc[df_neutros.index, "sub_sentimiento"] = df_neutros["sub_sentimiento"]
df.loc[df_neutros.index, "sub_confianza"] = df_neutros["sub_confianza"]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 512.45it/s, Materializing param=pooler.dense.weight]                               

BertModel LOAD REPORT from: C:\Users\GGNADI2\OneDrive - MAPFRE\Escritorio\Análisis_sentimiento\beto-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.bias              | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
BertModel LOAD REPORT from: C:\Users\GGNADI2\OneDrive - MAPFRE\Escritorio\Análisis_sentimiento\beto-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.bias              | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored w

Segunda vuelta análisis (había demasiados comentarios mal clasificados como neutros)

In [ ]:
def sentimiento_final(row):
    base = row.get("sentimiento_label", np.nan)
    sub = row.get("sub_sentimiento", np.nan)

    
    if base in ["Positivo", "Negativo"]:
        return base

    
    if sub == "Positivo_leve":
        return "Positivo"
    elif sub == "Negativo_leve":
        return "Negativo"
    else:
        return "Neutro"

df["sentimiento"] = df.apply(sentimiento_final, axis=1)

df["sentimiento"] = df["sentimiento"].fillna("Neutro")

In [ ]:
from sklearn.model_selection import train_test_split

# 70/30 train/test 
train_ratio_70 = 0.7
train_70, test_30 = train_test_split(
    df,
    train_size=train_ratio_70,
    test_size=1 - train_ratio_70,
    random_state=42,
    stratify=df["sentimiento"] if "sentimiento" in df.columns else None
)

# 30/70 train/test 
train_ratio_30 = 0.3
train_30, test_70 = train_test_split(
    df,
    train_size=train_ratio_30,
    test_size=1 - train_ratio_30,
    random_state=42,
    stratify=df["sentimiento"] if "sentimiento" in df.columns else None
)

print("Split shapes antes de ABSA:")
print("70/30 train:", train_70.shape)
print("70/30 test:", test_30.shape)
print("30/70 train:", train_30.shape)
print("30/70 test:", test_70.shape)

Split shapes antes de ABSA:
70/30 train: (159490, 76)
70/30 test: (68354, 76)
30/70 train: (68353, 76)
30/70 test: (159491, 76)


Análisis de aspectos (aplicado a cada split)

In [ ]:
import pandas as pd
from collections import defaultdict
from transformers import pipeline

RUTA_MODELO = r"C:\Users\GGNADI2\OneDrive - MAPFRE\Escritorio\Análisis_sentimiento\beto-sentiment"

device_id = 0 if torch.cuda.is_available() else -1

aspectos = {

    'Atencion_y_Trato': [
        'trato', 'buen trato', 'mal trato', 'trato correcto',
        'amabilidad', 'amable', 'desagradable', 'antipático',
        'empatía', 'empatia', 'falta de empatía',
        'atención', 'buena atención', 'mala atención',
        'atención deficiente', 'falta de atención',
        'profesionalidad', 'profesional', 'poco profesional',
        'explicaciones', 'explicación', 'explican bien', 'no explican',
        'información', 'información clara'
    ],

    'Gestion_del_Siniestro': [
        'siniestro', 'parte', 'abrir parte',
        'gestión', 'mala gestión', 'gestiones',
        'tramitación', 'trámite', 'trámites',
        'seguimiento', 'sin seguimiento',
        'expediente', 'cerrar expediente',
        'reclamación', 'reclamaciones',
        'documentación', 'papeles'
    ],

    'Reparacion_y_Profesionales': [
        'reparación', 'reparar', 'arreglo',
        'chapuza', 'mal arreglo', 'bien reparado',
        'profesional', 'técnico', 'operario',
        'fontanero', 'electricista', 'cerrajero',
        'albañil', 'pintor',
        'empresa reparadora',
        'no se presenta', 'no vino', 'plantón',
        'trabajo mal hecho', 'trabajo bien hecho'
    ],

    'Tiempos_y_Plazos': [
        'espera', 'tiempo de espera', 'larga espera',
        'mucho tiempo', 'tardan mucho',
        'demora', 'retraso', 'retrasos', 'tardanza',
        'días', 'semanas', 'meses',
        'urgente', 'urgencia',
        'rápido', 'rapidez', 'ágil',
        'lento', 'lentitud',
        'plazo', 'fuera de plazo'
    ],

    'Cobertura_y_Indemnizacion': [
        'cobertura', 'no cubre', 'no lo cubre',
        'cobertura insuficiente',
        'exclusiones', 'letra pequeña',
        'indemnización', 'indemnizar',
        'valoración', 'peritación',
        'perito', 'informe del perito',
        'importe', 'cantidad',
        'rechazo', 'denegado'
    ],

    'Precio_y_Poliza': [
        'precio', 'caro', 'muy caro', 'carísimo',
        'cuota', 'cuota alta', 'subida', 'subida de precio',
        'incremento', 'prima',
        'coste', 'coste elevado',
        'pago', 'pagar',
        'factura', 'cobro',
        'relación calidad precio',
        'renovación', 'anulación', 'baja'
    ],

    'Comunicacion_y_Canales': [
        'teléfono', 'llamar', 'no cogen', 'no responden',
        'contacto', 'difícil contactar',
        'email', 'correo', 'mensaje', 'sms',
        'respuesta', 'sin respuesta',
        'comunicación', 'mala comunicación',
        'robot', 'contestador',
        'app', 'aplicación',
        'web', 'página web',
        'oficina', 'sucursal'
    ]
}


aspect_model = pipeline(
    "sentiment-analysis",
    model=RUTA_MODELO,
    tokenizer=RUTA_MODELO,
    local_files_only=True,
    truncation=True,
    max_length=512,
    device=device_id,
    batch_size=32
)



Loading weights: 100%|██████████| 201/201 [00:00<00:00, 874.89it/s, Materializing param=classifier.weight]                              
BertForSequenceClassification LOAD REPORT from: C:\Users\GGNADI2\OneDrive - MAPFRE\Escritorio\Análisis_sentimiento\beto-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
BertForSequenceClassification LOAD REPORT from: C:\Users\GGNADI2\OneDrive - MAPFRE\Escritorio\Análisis_sentimiento\beto-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


def find_aspects(text, aspectos):
    text_lower = text.lower()
    return [category for category, keywords in aspectos.items() if any(keyword in text_lower for keyword in keywords)]

def apply_absa(data, aspectos, aspect_model, batch_size=32):
    """Aplica ABSA a un dataframe (train o test)"""
    data = data.copy()
    data["texto"] = data["ES - Corredores - Verbatim Comment including Corredores"].astype(str)
    data["aspect_matches"] = data["texto"].apply(lambda t: find_aspects(t, aspectos))

    rows = []
    for idx, aspects_list in data["aspect_matches"].items():
        text = data.at[idx, "texto"]
        for aspect in aspects_list:
            rows.append({
                "idx": idx,
                "aspect": aspect,
                "input_text": f"{aspect}. {text}"
            })

    predictions = []
    rows_texts = [row["input_text"] for row in rows]
    for i in range(0, len(rows_texts), batch_size):
        batch_texts = rows_texts[i:i + batch_size]
        predictions.extend(aspect_model(batch_texts))

    results = defaultdict(list)
    for row, pred in zip(rows, predictions):
        results[row["idx"]].append(str(row["aspect"]))

    data["aspectos_absa"] = data.index.map(lambda idx: results.get(idx, []))
    data = data.drop(columns=["aspect_matches"], errors="ignore")
    return data

train_70 = apply_absa(train_70, aspectos, aspect_model)
test_30 = apply_absa(test_30, aspectos, aspect_model)
train_30 = apply_absa(train_30, aspectos, aspect_model)
test_70 = apply_absa(test_70, aspectos, aspect_model)

df = apply_absa(df, aspectos, aspect_model)


In [ ]:
def find_aspects(text, aspectos):
    text_lower = text.lower()
    return [category for category, keywords in aspectos.items() if any(keyword in text_lower for keyword in keywords)]

def apply_absa(data, aspectos, aspect_model, batch_size=32):
    """Aplica ABSA a un dataframe (train o test)"""
    data = data.copy()
    data["texto"] = data["ES - Corredores - Verbatim Comment including Corredores"].astype(str)
    data["aspect_matches"] = data["texto"].apply(lambda t: find_aspects(t, aspectos))

    rows = []
    for idx, aspects_list in data["aspect_matches"].items():
        text = data.at[idx, "texto"]
        for aspect in aspects_list:
            rows.append({
                "idx": idx,
                "aspect": aspect,
                "input_text": f"{aspect}. {text}"
            })

    predictions = []
    rows_texts = [row["input_text"] for row in rows]
    for i in range(0, len(rows_texts), batch_size):
        batch_texts = rows_texts[i:i + batch_size]
        predictions.extend(aspect_model(batch_texts))

    results = defaultdict(list)
    for row, pred in zip(rows, predictions):
        results[row["idx"]].append(str(row["aspect"]))

    data["aspectos_absa"] = data.index.map(lambda idx: results.get(idx, []))
    data = data.drop(columns=["aspect_matches"], errors="ignore")
    
    return data

train_70 = apply_absa(train_70, aspectos, aspect_model)
test_30 = apply_absa(test_30, aspectos, aspect_model)
train_30 = apply_absa(train_30, aspectos, aspect_model)
test_70 = apply_absa(test_70, aspectos, aspect_model)


In [ ]:
for data_name, data in [("train_70", train_70), ("test_30", test_30), ("train_30", train_30), ("test_70", test_70), ("full", df)]:
    for aspecto_categoria in aspectos.keys():
        columna = f"aspecto_{aspecto_categoria}"
        data[columna] = data["aspectos_absa"].apply(
            lambda lista_aspectos: int(
                isinstance(lista_aspectos, list) and any(
                    (
                        isinstance(aspecto, dict) and aspecto.get("aspect") == aspecto_categoria
                    ) or (
                        isinstance(aspecto, str) and aspecto == aspecto_categoria
                    )
                    for aspecto in lista_aspectos
                )
            )
        )
    data["aspectos_count"] = data["aspectos_absa"].apply(
        lambda x: len(x) if isinstance(x, list) else 0
    )

train_70.to_excel("hogar_train_70_30.xlsx", index=False)
test_30.to_excel("hogar_test_70_30.xlsx", index=False)
train_30.to_excel("hogar_train_30_70.xlsx", index=False)
test_70.to_excel("hogar_test_30_70.xlsx", index=False)

df.to_excel("hogar_completo_analizado_absa.xlsx", index=False)

print("\nArchivos generados:")
print("- hogar_train_70_30.xlsx")
print("- hogar_test_70_30.xlsx")
print("- hogar_train_30_70.xlsx")
print("- hogar_test_30_70.xlsx")
print("- hogar_completo_analizado_absa.xlsx")
